Przetworzyć plik `/var/log/apache2/access.log`, wyciągnąć z niego istotne informacje statystyczne i zapisać raport do pliku CSV lub JSON.

- liczbę wszystkich żądań,
- liczbę unikalnych adresów IP,
- najczęściej odwiedzane zasoby (/index.html, /login, /img/logo.png itp.),
- rozkład kodów odpowiedzi HTTP (np. 200, 404, 500).

Program na koniec zapisze wyniki do pliku `raport.json` w formacie:

```
{
  "liczba_żądań": 2457,
  "unikalne_ip": 57,
  "top_zasoby": {
    "/": 432,
    "/login": 128,
    "/favicon.ico": 97
  },
  "kody_http": {
    "200": 2134,
    "404": 185,
    "500": 6
  }
}
```

Dodatkowo wyświetli 5 najczęściej odwiedzających adresów IP.

In [2]:
import re
import json
from collections import Counter

LOG_FILE = "/var/log/apache2/access.log"
OUTPUT = "raport.json"

pattern = re.compile(
    r'(?P<ip>\d+\.\d+\.\d+\.\d+).*?"(?:GET|POST) (?P<resource>[^ ]+) [^"]*" (?P<status>\d{3})'
)

ips = []
resources = []
statuses = []

with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        match = pattern.search(line)
        if match:
            ips.append(match.group("ip"))
            resources.append(match.group("resource"))
            statuses.append(match.group("status"))

count_requests = len(ips)
unique_ips = len(set(ips))

top_resources = Counter(resources).most_common(10)
top_statuses = Counter(statuses)
top_visitors = Counter(ips).most_common(5)
top_resources_dict = {res: cnt for res, cnt in top_resources}
top_statuses_dict = dict(top_statuses)

report = {
    "liczba_żądań": count_requests,
    "unikalne_ip": unique_ips,
    "top_zasoby": top_resources_dict,
    "kody_http": top_statuses_dict
}

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Raport zapisano do raport.json\n")

print("5 najczęściej odwiedzających IP:")
for ip, count in top_visitors:
    print(f"{ip} → {count} żądań")


Raport zapisano do raport.json

5 najczęściej odwiedzających IP:
149.156.84.120 → 4878 żądań
149.156.84.107 → 2184 żądań
80.49.174.190 → 1828 żądań
149.156.84.106 → 1673 żądań
139.28.41.143 → 599 żądań
